<a href="https://colab.research.google.com/github/krjeyy/AI-ML-lab-experiments/blob/main/id3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import numpy as np
import json

def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    probabilities = counts / counts.sum()
    return -np.sum(probabilities * np.log2(probabilities))

def information_gain(data, split_attribute_name, target_name="target"):
    total_entropy = entropy(data[target_name])
    vals, counts = np.unique(data[split_attribute_name], return_counts=True)

    weighted_entropy = sum(
        (counts[i] / counts.sum()) *
        entropy(data[data[split_attribute_name] == vals[i]][target_name])
        for i in range(len(vals))
    )

    return total_entropy - weighted_entropy

def id3(data, features, target_attribute_name="target",
        parent_node_class=None):

    if len(data) == 0:
        return parent_node_class

    if len(np.unique(data[target_attribute_name])) == 1:
        return data[target_attribute_name].iloc[0]

    if len(features) == 0:
        return data[target_attribute_name].mode()[0]

    parent_node_class = data[target_attribute_name].mode()[0]

    gains = [
        information_gain(data, feature, target_attribute_name)
        for feature in features
    ]

    best_feature = features[np.argmax(gains)]

    if max(gains) <= 0:
        return parent_node_class

    tree = {best_feature: {}}
    remaining_features = [f for f in features if f != best_feature]

    for value in np.unique(data[best_feature]):
        subset = data[data[best_feature] == value].drop(
            columns=[best_feature]
        )

        tree[best_feature][value] = id3(
            subset,
            remaining_features,
            target_attribute_name,
            parent_node_class
        )

    return tree

def predict(query, tree):
    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    value = query.get(feature)

    if value not in tree[feature]:
        return None

    return predict(query, tree[feature][value])

# Sample dataset
data = pd.DataFrame({
    'Outlook': [
        'Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain',
        'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny',
        'Overcast', 'Overcast', 'Rain'
    ],
    'Temperature': [
        'Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool',
        'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'
    ],
    'Humidity': [
        'High', 'High', 'High', 'High', 'Normal', 'Normal',
        'Normal', 'High', 'Normal', 'Normal', 'Normal',
        'High', 'Normal', 'High'
    ],
    'Wind': [
        'Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong',
        'Strong', 'Weak', 'Weak', 'Weak', 'Strong',
        'Strong', 'Weak', 'Strong'
    ],
    'target': [
        'No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes',
        'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No'
    ]
})

# Build the ID3 decision tree
features = list(data.columns[:-1])
tree = id3(data, features)

print("Generated Decision Tree:")
print(json.dumps(tree, indent=4))

# Make predictions
query1 = {
    'Outlook': 'Sunny',
    'Temperature': 'Hot',
    'Humidity': 'Normal',
    'Wind': 'Weak'
}

query2 = {
    'Outlook': 'Rain',
    'Temperature': 'Cool',
    'Humidity': 'Normal',
    'Wind': 'Weak'
}

print("\nPredicting for:", query1)
print("Prediction:", predict(query1, tree))

print("\nPredicting for:", query2)
print("Prediction:", predict(query2, tree))
